### **Statistical Factor Models**
April 2025

*Imports*

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import date

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

from model import *

/Users/tomcole/python/monthly-posts/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


*Data*

In [2]:
tickers = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'TSLA', 'NVDA', 'JPM', 'JNJ', 'V', 
    'PG', 'HD', 'MA', 'DIS', 'ADBE', 'PYPL', 'NFLX', 'CRM', 'BAC', 'INTC',
    'CSCO', 'PEP', 'KO', 'T', 'WMT', 'XOM', 'CVX', 'ABT', 'ABBV', 'LLY',
    'PFE', 'MRK', 'BMY', 'UNH', 'VZ', 'CMCSA', 'COST', 'ORCL', 'AVGO', 'QCOM',
    'TXN', 'AMD', 'NKE', 'SBUX', 'MDT', 'UPS', 'CAT', 'DE', 'MMM', 'HON',
    'IBM', 'GS', 'BLK', 'AXP', 'SPGI', 'RTX', 'BA', 'GE', 'LMT', 'NOW',
    'INTU', 'ISRG', 'SYK', 'ZTS', 'DHR', 'TMO', 'AMGN', 'GILD', 'REGN', 'VRTX',
    'ADI', 'MU', 'EA', 'ADP', 'FIS', 'SNAP', 'ZM', 'ROKU',
    'SHOP', 'DASH', 'UBER', 'LYFT', 'AAL', 'DAL', 'UAL', 'LUV', 'FDX',
    'NEE', 'SO', 'DUK', 'D', 'EXC', 'AEP', 'PLD', 'AMT', 'CCI', 'PSA','BK','CL','CI'
]

# Dates
start_date = date(2023,1,1)
end_date = date(2025,1,1)

# Download Data
data = yf.download(tickers,start = start_date,end = end_date)['Close']

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  100 of 100 completed


In [3]:
returns = data.pct_change(axis = 0).dropna()
R = returns.T

### **Estimating the Statistical Factor Model**

**A first approach**

In [4]:
U, S, Vt = np.linalg.svd(R, full_matrices=False)
B = U
F = np.diag(S) @ Vt

#### **1. Identifying Factors via PCA**

In [5]:
# Parameters
tau_s = 120
tau_f = 60
p = 5
m = 3
T_max = 252

In [6]:
idio_returns = []
factor_returns = []
factor_loadings = []
for t in range(T_max, R.shape[1]):
    
    # 1. Estimate B_{t-1} using data up to t-1
    R_window_t_minus_1 = R.iloc[:, t - T_max:t]
    B_t_minus_1, W_sigma = statistical_factor_model(R_window_t_minus_1, tau_s, tau_f, p, m)
    W2 = W_sigma @ W_sigma
    r_t = R.iloc[:, t].values
    
    # 2. Estimate f_t using B_{t-1}
    f_hat_t = np.linalg.pinv(B_t_minus_1.T @ W2 @ B_t_minus_1) @ B_t_minus_1.T @ W2 @ r_t
    factor_returns.append(f_hat_t)

    # 3. Estimate B_t using data up to t
    R_window_t = R.iloc[:, t - T_max + 1:t + 1]
    B_t, _ = statistical_factor_model(R_window_t, tau_s, tau_f, p, m)
    factor_loadings.append(B_t)

    # 4. Compute residual using B_t
    e_t = r_t - B_t @ f_hat_t
    idio_returns.append(e_t)


factor_returns_pca = np.vstack(factor_returns)
idio_returns_pca = np.vstack(idio_returns)
factor_loadings_pca = factor_loadings.copy()

#### **2. Identifying Factors via IPCA**

In [7]:
idio_returns = []
factor_returns = []
factor_loadings = []
for t in range(T_max, R.shape[1]):
    
    # 1. Estimate B_{t-1} using data up to t-1
    R_window_t_minus_1 = R.iloc[:, t - T_max:t]
    B_t_minus_1, W_sigma = statistical_factor_model_ica(R_window_t_minus_1, tau_s, tau_f, p, m)
    W2 = W_sigma @ W_sigma
    
    r_t = R.iloc[:, t].values
    
    # 2. Estimate f_t using B_{t-1}
    f_hat_t = np.linalg.pinv(B_t_minus_1.T @ W2 @ B_t_minus_1) @ B_t_minus_1.T @ W2 @ r_t
    factor_returns.append(f_hat_t)

    # 3. Estimate B_t using data up to t
    R_window_t = R.iloc[:, t - T_max + 1:t + 1]
    B_t, _ = statistical_factor_model_ica(R_window_t, tau_s, tau_f, p, m)
    factor_loadings.append(B_t)
    # 4. Compute residual using B_t
    e_t = r_t - B_t @ f_hat_t
    idio_returns.append(e_t)

factor_returns_ipca = np.vstack(factor_returns)
idio_returns_ipca = np.vstack(idio_returns)
factor_loadings_ipca = factor_loadings.copy()

#### **3. Identifying Factors via SPCA**

In [8]:
idio_returns = []
factor_returns = []
factor_loadings = []
for t in range(T_max, R.shape[1]):
    # 1. Estimate B_{t-1} using data up to t-1
    R_window_t_minus_1 = R.iloc[:, t - T_max:t]
    B_t_minus_1, W_sigma = statistical_factor_model_spca(R_window_t_minus_1, tau_s, tau_f, p, m)
    W2 = W_sigma @ W_sigma
    
    r_t = R.iloc[:, t].values
    
    # 2. Estimate f_t using B_{t-1}
    f_hat_t = np.linalg.pinv(B_t_minus_1.T @ W2 @ B_t_minus_1) @ B_t_minus_1.T @ W2 @ r_t
    factor_returns.append(f_hat_t)

    # 3. Estimate B_t using data up to t
    R_window_t = R.iloc[:, t - T_max + 1:t + 1]
    B_t, _ = statistical_factor_model_spca(R_window_t, tau_s, tau_f, p, m)
    factor_loadings.append(B_t)

    # 4. Compute residual using B_t
    e_t = r_t - B_t @ f_hat_t
    idio_returns.append(e_t)

factor_returns_spca = np.vstack(factor_returns)
idio_returns_spca = np.vstack(idio_returns)
factor_loadings_spca = factor_loadings.copy()

### **Results**

*Plot: Factor Returns*

In [9]:
fig = make_subplots(rows=2, cols=3,
                    subplot_titles=['PCA','SPCA','IPCA','','',''],
                    vertical_spacing=0.08, horizontal_spacing=0.08)

factor_returns = {'PCA': factor_returns_pca, 'SPCA': factor_returns_spca, 'IPCA': factor_returns_ipca}

factor_colors = px.colors.qualitative.Plotly  

for col_idx, method in enumerate(factor_returns):
    for factor_idx in range(factor_returns[method].shape[1]):
        
        color = factor_colors[factor_idx % len(factor_colors)]
        
        # Daily Returns
        fig.add_trace(
            go.Scatter(
                x = data.index[T_max+1:],
                y=factor_returns[method][:, factor_idx],
                name=f'{method}-Factor {factor_idx}',
                legendgroup=method,  
                legendgrouptitle_text=method,
                showlegend=True,
                line=dict(color=color),
                mode='lines'
            ),
            row=1,
            col=col_idx + 1,
        )

        # Cumulative Returns
        fig.add_trace(
            go.Scatter(
                x = data.index[T_max+1:],
                y=(1 + factor_returns[method][:, factor_idx]).cumprod(),
                name=f'{method}-Factor {factor_idx}',
                legendgroup=method,  
                showlegend=False,    
                line=dict(color=color),
                mode='lines'
            ),
            row=2,
            col=col_idx + 1,
        )

fig.update_layout(
    legend=dict(
        tracegroupgap=10,  
        groupclick="toggleitem",  
    ),
    # template='plotly_white'
)

fig.show()

*Plot: Idio Returns*

In [10]:
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=['PCA','SPCA','IPCA'],
                    vertical_spacing=0.08, horizontal_spacing=0.08)

idio_returns = {'PCA': idio_returns_pca, 'SPCA': idio_returns_spca, 'IPCA': idio_returns_ipca}

asset_colors = px.colors.qualitative.Plotly  

for col_idx, method in enumerate(idio_returns):
    for asset_idx in range(idio_returns[method].shape[1]):
        
        color = asset_colors[asset_idx % len(asset_colors)]
        
        # Daily Returns
        fig.add_trace(
            go.Scatter(
                x = data.index[T_max+1:],
                y= (1 + idio_returns[method][:, asset_idx]).cumprod(),
                name=data.columns[asset_idx],
                showlegend = (col_idx == 0),
                line=dict(color=color),
                mode='lines'
            ),
            row=1,
            col=col_idx + 1,
        )

fig.show()

*Plot: Factor Loadings*

In [13]:
import plotly.subplots as sp
import plotly.graph_objects as go
import pandas as pd
import numpy as np

loadings_pca = factor_loadings_pca[-1]
loadings_spca = factor_loadings_spca[-1]  
loadings_ipca = factor_loadings_ipca[-1]  

methods = {
    'PCA': loadings_pca,
    'SPCA': loadings_spca, 
    'IPCA': loadings_ipca
}
factor_names = ['Factor 0', 'Factor 1', 'Factor 2']
asset_names = data.columns  

fig = sp.make_subplots(
    rows=3, cols=3,
    subplot_titles=[f"{method} - {factor}" 
                  for method in methods 
                  for factor in factor_names],
    vertical_spacing=0.1,
    horizontal_spacing=0.05,
    shared_yaxes=True
)

colors = px.colors.qualitative.Plotly[:3]

for row_idx, (method, loadings) in enumerate(methods.items(), 1):
    for col_idx, factor_idx in enumerate(range(3), 1):
        weights = loadings[:, factor_idx]
        
        fig.add_trace(
            go.Bar(
                x=asset_names,
                y=weights,
                name=factor_names[factor_idx],
                marker_color=colors[factor_idx],
                showlegend=(row_idx == 1)
            ),
            row=row_idx,
            col=col_idx
        )

# Update layout
fig.update_layout(
    height=900,
    width=1200,
    barmode='group',
    legend_title_text='Factors',
    margin=dict(l=50, r=50, b=100, t=50)
)

# Format x-axes (show every 10th asset)
for i in range(1, 10):
    fig.update_xaxes(
        tickmode='array',
        tickvals=np.arange(0, 100, 10),
        ticktext=[asset_names[i] for i in np.arange(0, 100, 10)],
        tickangle=45,
        row=(i-1)//3 + 1,
        col=(i-1)%3 + 1
    )

fig.show()